# Human-in-the-Loop: Előzetes Akciókapuk, Kockázati Szintbesorolás és Audit Naplózás

A lecke README-je bemutatja a Human-in-the-Loop-ot egy rövid részlettel, amely a felhasználótól azt kéri, hogy `APPROVE` vagy `REJECT` a válasz kézhezvétele után. Ez a minta jó kiindulópont, de az éles HITL megvalósítások általában három további elemet igényelnek:

1. Egy **előzetes akciókapu**, amely **az agent kockázatos lépése előtt** fut, hogy a költség, a visszafordíthatatlanság és a késleltetés kontroll alatt maradjon.
2. **Kockázati szintbesorolás**, hogy az alacsony kockázatú műveletek automatikusan végrehajtódjanak, a közepes kockázatú műveletek csoportosan jóváhagyhatók legyenek, és csak a magas kockázatú műveletek blokkoljanak egy embert.
3. Egy **audit napló és felülvizsgálati ciklus**, hogy minden kapu döntés JSONL formátumban legyen rögzítve, és egy elutasítás strukturált okkal újrapromtolja az agent-et ahelyett, hogy csak a `Revising...` szöveget írná ki.

Ez a jegyzetfüzet ezeket az elemeket azonos primitívekre építve fejleszti a `06-system-message-framework.ipynb`-vel. Teljes végigfutást biztosít `DEMO_MODE = True` (nincs szükség interaktív bevitelre), vagy valódi `input()` promptokkal, ha `DEMO_MODE = False`. Megjegyzés: DEMO_MODE-ban a harmadik cél újrapróbálása szkriptezve van, így a ciklus mechanikája végig látható. Valós, felülvizsgálaton alapuló újraosztályozás `DEMO_MODE = False`-t és egy operátort igényel.

**Nem része a tárgyalt témakörnek (más leckékben kezelve):** hitelesítés és hozzáférés-vezérlés (06-os lecke README fenyegetés 2), eszköz-hívó middleware (14-es lecke MAF mélyreható elemzés), több-agent vitázó minták.

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## Minta 1: Előzetes műveleti kapu

A README HITL részlete először hívja az ügynököt, majd kéri a felhasználót, hogy hagyja jóvá a kimenetet. Ez egy **utólagos műveleti** folyamat. Az ügynök már végrehajtotta a műveletet, így az LLM-hívás díja már megfizetésre került, és bármilyen mellékhatás (elküldött e-mail, írt adatbázissor, közzétett hozzászólás) már megtörtént.

Egy **előzetes műveleti** folyamat a kaput az ügynök kockázatos lépése elé illeszti be. Az ügynök javasolja a műveletet, a kapu eldönti, hogy végrehajtja-e, és csak jóváhagyás esetén következik be a mellékhatás.

| Aspektus | Utólagos jóváhagyás (README részlet) | Előzetes műveleti kapu (ez a jegyzetfüzet) |
|---|---|---|
| Mikor történik a jóváhagyás? | Miután az ügynök előállította a kimenetet | Mielőtt bármilyen mellékhatás végrehajtódik |
| LLM költsége elutasítás esetén | Már kifizetve | Csak a javaslatért fizetve, nem a műveletért |
| Visszafordíthatatlan mellékhatások | Lehetséges (a művelet már megtörtént) | Megakadályozva |
| Ellenőrzés átláthatósága | A jóváhagyás egy nyomtatott állítás | A jóváhagyás egy JSON rekord időbélyeggel, művelettel, indokkal |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## 2. mintázat: Kockázati szintezés

Nem minden művelet igényel emberi jóváhagyást. Egy nyilvános API ellenőrzése csak olvasásra más tétekkel jár, mint egy ügyfélnek küldött e-mail. Ha mindkettőt ugyanúgy kezeljük, az pazarlás az operátor figyelmével és lassítja az ügynököt.

Egy egyszerű, 3 szintű modell:

| Szint | Példák | Jóváhagyási folyamat |
|---|---|---|
| `alacsony` (csak olvasható) | Tudásbázis keresése, járatopciók lekérdezése, nyilvános weboldal betöltése | Automatikus végrehajtás, auditálva naplózva |
| `közepes` (olcsó módosítás) | Eredmény gyorsítótárazása, számláló növelése, emlékeztető ütemezése | Automatikus végrehajtás, de napi csoportos felülvizsgálat |
| `magas` (külső felé irányuló vagy visszafordíthatatlan) | E-mail küldése, kártyalehúzás, nyilvános csatornára posztolás | Emberi jóváhagyásra várakozás |

Ez egy szint felosztás. A termelési rendszerek gyakran használnak részletesebb szinteket (pl. AWS IAM jogosultsági szintek, szerepkör alapú hozzáférési szintek). Az alábbi három szintű verzió a legkisebb hasznos verzió egy olyan ügynök számára, amely keveri a csak olvasó és mellékhatásokat okozó műveleteket.

Az alábbi osztályozó kulcsszó heuristikákat alkalmaz, így a demó determinisztikus és olcsó marad. Egy termelési rendszerben ezt egy tanult osztályozóra vagy egy szabályrendszerre cserélnénk.

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Minta 3: Audit napló és felülvizsgálati ciklus

Egy `print("Response approved.")` nem audit napló. A bizalom érdekében minden kapu döntést strukturált eseményként kell rögzíteni, amelyet később lekérdezhet, lejátszhat vagy egy incidens átvizsgáláshoz csatolhat.

Két rész:

1. **Csak hozzáfűzhető JSONL.** Egy sor döntésenként, időbélyeggel, művelettel, szinttel, döntéssel, indoklással. Könnyen kereshető, később könnyen továbbítható egy valódi napló tárolóba.
2. **Felülvizsgálati ciklus elutasítás esetén.** Amikor a kapu `deny` értéket ad vissza, az ügynök újra lekérdezi magát az elutasítás okával a kontextusban, így a következő javaslat elkerülheti a problémát.

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## További források

Számos más nyilvános projekt valósít meg ezeknek a HITL mintáknak változatait. Hasonlítsa össze a megközelítéseket, hogy megtalálja a stackjéhez illőt:

- **LangChain** emberi beavatkozást igénylő eszközcsomagok ([dokumentáció](https://python.langchain.com/docs/integrations/tools/human_tools)): olyan eszközcsomagok, amelyek leállítják a végrehajtást emberi bevitelhez.
- **AutoGen** `UserProxyAgent` ([v0.2 dokumentáció](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+-ban átstrukturálták): egy agent szerep használata, amely kifejezetten az embert képviseli többagentés beszélgetésekben.
- **Semantic Kernel** függvényszűrők ([dokumentáció](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters)): middleware, amely minden függvényhívás körül fut, alkalmas döntéskapu logika megvalósítására.

Minden projekt eltérően kezeli a három al-mintát: a LangChain eszközökként burkolja be ezeket, az AutoGen agent szerepet használ, a Semantic Kernel middleware szűrőket használ. Olvasson el egy vagy két megvalósítást végig, mielőtt kiválasztja a saját agentje számára a dizájnt.

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:
Ez a dokumentum az AI fordítási szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár az pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum az anyanyelvén tekintendő hiteles forrásnak. Fontos információk esetén professzionális emberi fordítást javasolunk. Nem vállalunk felelősséget semmilyen félreértésért vagy téves értelmezésért, amely ebből a fordításból ered.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
